> **Run:** Pilot
>
> **Evidence label:** DEVELOPMENT / PRELIMINARY
>
> **Original file:** `text2sql_execution_outcome_bias_pilot_v2_reviewed.ipynb`
>
> **Purpose:** holding the natural-language question, schema, and candidate SQL fixed, does changing only the reported execution outcome (`no feedback` vs `empty` vs `non-empty`) systematically change an LLM's judgment of SQL correctness?
>
> **Drive output folder:** `Text2SQL_ExecutionBias_Pilot`
>
> This notebook is not part of the confirmatory protocol.

# Pilot — Execution-Outcome Bias in Text-to-SQL Verification

**Research question:** holding the natural-language question, schema, and candidate SQL fixed, does changing only the reported execution outcome (`no feedback` vs `empty` vs `non-empty`) systematically change an LLM's judgment of SQL correctness?

This reviewed version automatically:

- installs dependencies with conservative version bounds;
- detects GPU / VRAM;
- locates a local BIRD/Spider-style benchmark, or downloads an executable Spider fallback;
- builds **50 base questions**;
- creates one **correct** and one **controlled-wrong** candidate per base question;
- **filters out SQL for which empty/non-empty feedback is logically implausible** (e.g. plain scalar `COUNT(*)`);
- compares SQL results in an **order-aware way only when ORDER BY matters**, otherwise as multisets;
- uses real wall-clock SQLite timeouts and rejects truncated execution comparisons;
- expands each candidate into **3 counterfactual feedback conditions**;
- runs models sequentially with caching/resume;
- computes paired effects for:
  - **empty-induced rejection of correct SQL**;
  - **non-empty false reassurance for wrong SQL**;
  - optional **harmful correction** after a revision is generated.

Default primary models:

- `Qwen/Qwen3-8B`
- `google/gemma-3-12b-it`

If Gemma cannot be loaded, the notebook automatically tries:

- `mistralai/Mistral-7B-Instruct-v0.3`

> **Pilot principle:** the main experiment intentionally uses counterfactual feedback. Question, schema, and candidate SQL are identical across treatment conditions; only the reported execution outcome changes.

In [1]:
# ============================================================
# 0. AUTOSETUP
# ============================================================
import os, sys, subprocess, random, time, warnings
from pathlib import Path

def pip_install(pkgs):
    # Avoid blanket upgrades; only change packages that do not satisfy constraints.
    cmd = [sys.executable, '-m', 'pip', 'install', '-q', '--upgrade-strategy', 'only-if-needed', *pkgs]
    subprocess.check_call(cmd)

REQ = [
    'transformers>=4.51.0,<5.0',
    'accelerate>=1.2.0',
    'bitsandbytes>=0.45.0',
    'huggingface_hub>=0.27.0',
    'sqlglot>=26.0.0',
    'pandas>=2.0.0',
    'numpy>=1.26.0,<2.4',
    'scipy>=1.11.0',
    'tqdm>=4.66.0',
    'sentencepiece>=0.2.0',
]
print('Installing/checking dependencies...')
pip_install(REQ)

import torch, pandas as pd, numpy as np, sqlglot, transformers
from tqdm.auto import tqdm

print('=' * 72)
print('Python       :', sys.version.split()[0])
print('Torch        :', torch.__version__)
print('Transformers :', transformers.__version__)
print('SQLGlot      :', sqlglot.__version__)
print('CUDA         :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU          :', torch.cuda.get_device_name(0))
    props = torch.cuda.get_device_properties(0)
    print('VRAM         :', round(props.total_memory / 1024**3, 1), 'GB')
else:
    print('WARNING: no CUDA GPU detected; local 7B/8B/12B inference will be slow.')
print('=' * 72)

Installing/checking dependencies...
Python       : 3.13.15
Torch        : 2.11.0+cpu
Transformers : 4.57.6
SQLGlot      : 30.18.0
CUDA         : False


In [2]:
# ============================================================
# 1. CONFIG — GOOGLE DRIVE PERSISTENT VERSION
# ============================================================
from pathlib import Path
import os, random, numpy as np

# ------------------------------------------------------------
# AUTO-MOUNT GOOGLE DRIVE IF RUNNING IN COLAB
# ------------------------------------------------------------
IN_COLAB = Path("/content").exists()

if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)

        DRIVE_ROOT = Path(
            "/content/drive/MyDrive/Text2SQL_ExecutionBias_Pilot"
        )
        DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

        print("✓ Google Drive mounted")
        print("✓ Persistent project folder:", DRIVE_ROOT)

    except Exception as e:
        raise RuntimeError(
            f"Google Drive mount failed: {e}"
        )
else:
    # Non-Colab fallback.
    DRIVE_ROOT = Path("./Text2SQL_ExecutionBias_Pilot")
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

    print("Not running in Colab.")
    print("Using local persistent folder:", DRIVE_ROOT.resolve())


CFG = {
    # ========================================================
    # PILOT DESIGN
    # ========================================================
    "seed": 42,

    # 50 base questions
    # -> 50 correct + 50 wrong candidates
    # -> 100 candidates
    # -> 300 trials/model
    "n_base": 50,

    "conditions": [
        "none",
        "empty",
        "nonempty",
    ],

    # ========================================================
    # DATASET
    # ========================================================

    # Optional manual benchmark path.
    #
    # Example BIRD on Drive:
    # "/content/drive/MyDrive/BIRD"
    #
    # Leave None for automatic discovery / Spider fallback.
    "data_root": None,

    "auto_download_spider": True,

    # IMPORTANT:
    # ALL outputs now persist directly to Google Drive.
    "work_dir": str(DRIVE_ROOT),

    # ========================================================
    # MODELS
    # ========================================================
    "models": [
        "Qwen/Qwen3-8B",
        "google/gemma-3-12b-it",
    ],

    # Automatically used if Gemma is gated/unavailable.
    "fallback_models": [
        "mistralai/Mistral-7B-Instruct-v0.3",
    ],

    "minimum_successful_models": 2,
    "skip_model_on_load_error": True,

    # ========================================================
    # INFERENCE
    # ========================================================
    "max_new_tokens": 220,
    "temperature": 0.0,

    # Recommended for Colab GPU.
    "load_in_4bit": True,

    "trust_remote_code": False,

    # ========================================================
    # SQL VALIDATION
    # ========================================================
    "gold_timeout_s": 20,
    "mutation_timeout_s": 20,

    "max_result_rows": 10000,
    "max_schema_chars": 30000,

    # Increase if it struggles to find 50 valid examples.
    "max_candidate_search": 2200,

    # Exclude queries like plain COUNT(*)
    # where "0 rows" is logically impossible.
    "require_outcome_flexible": True,

    # ========================================================
    # EXPERIMENT
    # ========================================================
    "shuffle_trials": True,

    # FALSE = resume existing results from Google Drive.
    #
    # Keep False unless you explicitly want to restart.
    "overwrite_results": False,
}


# ------------------------------------------------------------
# INITIALIZE
# ------------------------------------------------------------
WORK = Path(CFG["work_dir"])
WORK.mkdir(parents=True, exist_ok=True)

random.seed(CFG["seed"])
np.random.seed(CFG["seed"])


# ------------------------------------------------------------
# HUGGING FACE TOKEN
# ------------------------------------------------------------
HF_TOKEN = (
    os.environ.get("HF_TOKEN")
    or os.environ.get("HUGGINGFACE_TOKEN")
    or os.environ.get("HUGGING_FACE_HUB_TOKEN")
)


print("\n" + "=" * 72)
print("Pilot configuration")
print("=" * 72)
print("WORK directory    :", WORK.resolve())
print("Google Drive      :", IN_COLAB)
print("HF token detected :", bool(HF_TOKEN))
print("Base questions    :", CFG["n_base"])
print("Trials / model    :", CFG["n_base"] * 2 * len(CFG["conditions"]))
print("=" * 72)

if not HF_TOKEN:
    print(
        "\nNOTE:"
        "\n- Qwen and Mistral are public."
        "\n- Gemma may require HF_TOKEN + accepted Hugging Face license."
        "\n- If Gemma fails, the notebook will automatically try Mistral."
    )

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ Google Drive mounted
✓ Persistent project folder: /content/drive/MyDrive/Text2SQL_ExecutionBias_Pilot

Pilot configuration
WORK directory    : /content/drive/MyDrive/Text2SQL_ExecutionBias_Pilot
Google Drive      : True
HF token detected : False
Base questions    : 50
Trials / model    : 300

NOTE:
- Qwen and Mistral are public.
- Gemma may require HF_TOKEN + accepted Hugging Face license.
- If Gemma fails, the notebook will automatically try Mistral.


## 1. Dataset discovery

The notebook prefers an already-mounted executable BIRD/Spider dataset. It scans common notebook roots for:

- a `dev.json`/similar file with `question`, `db_id`, and gold SQL;
- `.sqlite` databases whose filename matches the `db_id`.

If nothing usable is found and `auto_download_spider=True`, it downloads a Spider SQLite bundle from Hugging Face (~100 MB).

In [3]:
# ============================================================
# 2. LOCATE / DOWNLOAD EXECUTABLE BENCHMARK
# ============================================================
import json, zipfile
from huggingface_hub import hf_hub_download

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def looks_like_examples(path):
    try:
        x = load_json(path)
        if not isinstance(x, list) or not x:
            return False
        r = x[0]
        return (
            isinstance(r, dict)
            and "db_id" in r
            and "question" in r
            and any(k in r for k in ["SQL", "query", "sql"])
        )
    except Exception:
        return False

def search_roots():
    roots = []
    if CFG["data_root"]:
        roots.append(Path(CFG["data_root"]))
    for p in ["/kaggle/input", "/content", "/mnt/data", "."]:
        pp = Path(p)
        if pp.exists():
            roots.append(pp)
    out, seen = [], set()
    for r in roots:
        key = str(r.resolve())
        if key not in seen:
            seen.add(key)
            out.append(r)
    return out

def sqlite_index(roots):
    idx = {}
    for root in roots:
        try:
            for p in root.rglob("*.sqlite"):
                idx.setdefault(p.stem, []).append(p)
        except Exception:
            pass
    return idx

def discover():
    roots = search_roots()
    print("Searching:", [str(r) for r in roots])
    idx = sqlite_index(roots)
    print("SQLite DB ids indexed:", len(idx))

    jsons = []
    for root in roots:
        try:
            for p in root.rglob("*.json"):
                n = p.name.lower()
                if n in {"dev.json", "validation.json", "train_spider.json", "train.json"} or "dev" in n:
                    jsons.append(p)
        except Exception:
            pass

    jsons = sorted(
        set(jsons),
        key=lambda p: (
            0 if p.name.lower() == "dev.json" else 1,
            0 if any(x in str(p).lower() for x in ["bird", "spider"]) else 1,
            len(str(p)),
        ),
    )

    for jp in jsons:
        if not looks_like_examples(jp):
            continue
        rows = load_json(jp)
        match = sum(str(r.get("db_id", "")) in idx for r in rows[:100])
        if match:
            print("Found:", jp, "| DB matches in first 100:", match)
            return jp, idx
    return None, idx

EXAMPLE_JSON, SQLITE_INDEX = discover()

if EXAMPLE_JSON is None and CFG["auto_download_spider"]:
    target = WORK / "spider_bundle"
    dev_path = target / "spider_data" / "dev.json"
    if not dev_path.exists():
        print("No local executable benchmark found.")
        print("Downloading Spider SQLite bundle (~100 MB)...")
        target.mkdir(parents=True, exist_ok=True)
        zip_path = hf_hub_download(
            repo_id="HAL-9001/spider-databases",
            filename="spider_data.zip",
            repo_type="dataset",
            local_dir=str(target),
            token=HF_TOKEN,
        )
        with zipfile.ZipFile(zip_path, "r") as zf:
            zf.extractall(target)
    CFG["data_root"] = str(target)
    EXAMPLE_JSON, SQLITE_INDEX = discover()

if EXAMPLE_JSON is None:
    raise FileNotFoundError(
        "No executable BIRD/Spider dataset found. "
        "Set CFG['data_root'] to a folder containing example JSON + SQLite databases."
    )

RAW_ROWS = load_json(EXAMPLE_JSON)
print("Example JSON:", EXAMPLE_JSON)
print("Rows:", len(RAW_ROWS))

Searching: ['/kaggle/input', '/content']
SQLite DB ids indexed: 422
Found: /content/execution_bias_pilot_v2_reviewed/spider_bundle/spider_data/dev.json | DB matches in first 100: 100
Example JSON: /content/execution_bias_pilot_v2_reviewed/spider_bundle/spider_data/dev.json
Rows: 1034


In [4]:
# ============================================================
# 3. NORMALIZE EXAMPLES + SQLITE HELPERS
# ============================================================
import sqlite3, math, time
from collections import Counter

def normalize_example(r):
    sql = r.get('SQL') or r.get('query') or r.get('sql')
    return {
        'db_id': str(r['db_id']),
        'question': str(r['question']).strip(),
        'gold_sql': str(sql).strip(),
        'evidence': str(r.get('evidence', '') or '').strip(),
    }

def _path_distance(p, anchor):
    p_parts = Path(p).resolve().parts
    a_parts = Path(anchor).resolve().parts
    common = 0
    for x, y in zip(p_parts, a_parts):
        if x != y:
            break
        common += 1
    return (-common, len(p_parts))

def resolve_db(db_id):
    opts = SQLITE_INDEX.get(db_id, [])
    if not opts:
        raise FileNotFoundError(db_id)
    anchor = Path(EXAMPLE_JSON).parent
    opts = sorted(
        opts,
        key=lambda p: (
            0 if p.name == f'{db_id}.sqlite' else 1,
            *_path_distance(p, anchor),
        ),
    )
    return opts[0]

def norm_cell(v):
    if isinstance(v, bytes):
        return v.decode('utf-8', errors='replace')
    if isinstance(v, float) and math.isnan(v):
        return None
    return v

def execute_sql(db_path, sql, timeout_s=20, max_rows=None):
    if max_rows is None:
        max_rows = CFG['max_result_rows']

    con = sqlite3.connect(f'file:{db_path}?mode=ro', uri=True)
    started = time.monotonic()

    def progress():
        return 1 if (time.monotonic() - started) > timeout_s else 0

    con.set_progress_handler(progress, 1000)
    try:
        cur = con.cursor()
        cur.execute(sql)
        rows = cur.fetchmany(max_rows + 1)
        truncated = len(rows) > max_rows
        rows = rows[:max_rows]
        rows = [tuple(norm_cell(x) for x in row) for row in rows]
        return {
            'ok': True,
            'rows': rows,
            'n_rows': len(rows),
            'truncated': truncated,
            'error': None,
        }
    except Exception as e:
        return {
            'ok': False,
            'rows': None,
            'n_rows': None,
            'truncated': False,
            'error': str(e),
        }
    finally:
        con.close()

def _stable_cell(v):
    if v is None:
        return ('NULL', '')
    return (type(v).__name__, repr(v))

def _row_key(row):
    return tuple(_stable_cell(v) for v in row)

def has_top_level_order(sql):
    try:
        tree = sqlglot.parse_one(sql, read='sqlite')
        return bool(tree.args.get('order'))
    except Exception:
        return 'ORDER BY' in sql.upper()

def results_equivalent(a, b, sql_a, sql_b):
    if not a['ok'] or not b['ok']:
        return None
    if a.get('truncated') or b.get('truncated'):
        return None

    a_keys = [_row_key(r) for r in a['rows']]
    b_keys = [_row_key(r) for r in b['rows']]

    if has_top_level_order(sql_a) or has_top_level_order(sql_b):
        return a_keys == b_keys
    return Counter(a_keys) == Counter(b_keys)

def schema_text(db_path, max_chars=30000):
    con = sqlite3.connect(f'file:{db_path}?mode=ro', uri=True)
    try:
        cur = con.cursor()
        tables = [
            x[0] for x in cur.execute(
                "SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%' ORDER BY name"
            ).fetchall()
        ]
        blocks = []
        for t in tables:
            qt = '"' + t.replace('"', '""') + '"'
            cols = cur.execute(f'PRAGMA table_info({qt})').fetchall()
            col_s = ', '.join(
                f"{c[1]} {c[2] or ''}".strip() + (' PRIMARY KEY' if c[5] else '')
                for c in cols
            )
            blocks.append(f'TABLE {t} ({col_s});')
            for fk in cur.execute(f'PRAGMA foreign_key_list({qt})').fetchall():
                blocks.append(f'FOREIGN KEY {t}.{fk[3]} -> {fk[2]}.{fk[4]}')
        txt = '\n'.join(blocks)
        return txt if len(txt) <= max_chars else txt[:max_chars] + '\n-- [schema truncated]'
    finally:
        con.close()

EXAMPLES = []
for r in RAW_ROWS:
    try:
        ex = normalize_example(r)
        ex['db_path'] = str(resolve_db(ex['db_id']))
        EXAMPLES.append(ex)
    except Exception:
        pass

print('Examples with resolvable DB:', len(EXAMPLES))

Examples with resolvable DB: 1034


## 2. Building trustworthy wrong candidates

A mutation is accepted only when:

1. the gold SQL executes;
2. the mutated SQL executes;
3. it differs from the gold SQL;
4. **its result differs from the gold result on the same source database**.

This is a conservative pilot filter: it prevents us from labeling a trivially execution-equivalent rewrite as “wrong.”

In [5]:
# ============================================================
# 4. CONTROLLED SQL MUTATIONS + OUTCOME-FLEXIBILITY FILTER
# ============================================================
from sqlglot import exp
import hashlib

def to_sql(tree):
    return tree.sql(dialect='sqlite')

def _nearest_select(node):
    try:
        return node.find_ancestor(exp.Select)
    except Exception:
        return None

def outcome_flexible(sql):
    """True only when both 0-row and >0-row outcomes are logically plausible."""
    try:
        tree = sqlglot.parse_one(sql, read='sqlite')
    except Exception:
        return False

    # Keep the pilot simple: no top-level UNION/INTERSECT/EXCEPT.
    if not isinstance(tree, exp.Select):
        return False

    # SELECT 1 and other no-FROM scalar statements cannot be empty.
    if tree.args.get('from') is None and tree.args.get('from_') is None:
        return False

    # LIMIT 0 can never be non-empty.
    limit = tree.args.get('limit')
    if limit is not None:
        lim_expr = getattr(limit, 'expression', None)
        if isinstance(lim_expr, exp.Literal) and not lim_expr.is_string:
            try:
                if float(lim_expr.this) == 0:
                    return False
            except Exception:
                pass

    has_group = tree.args.get('group') is not None
    has_having = tree.args.get('having') is not None

    # Plain global aggregates (COUNT/MAX/AVG/...) generally return one row.
    # HAVING may suppress the global aggregate row, so HAVING cases are allowed.
    if not has_group and not has_having:
        for agg in tree.find_all(exp.AggFunc):
            if _nearest_select(agg) is tree:
                return False

    return True

def mutation_candidates(sql):
    out = []
    try:
        base = sqlglot.parse_one(sql, read='sqlite')
    except Exception:
        return out

    cmp_map = {
        exp.EQ: exp.NEQ,
        exp.NEQ: exp.EQ,
        exp.GT: exp.LTE,
        exp.GTE: exp.LT,
        exp.LT: exp.GTE,
        exp.LTE: exp.GT,
    }
    for old_t, new_t in cmp_map.items():
        nodes = list(base.find_all(old_t))
        for i in range(min(2, len(nodes))):
            t = base.copy()
            target = list(t.find_all(old_t))[i]
            try:
                target.replace(new_t(this=target.this.copy(), expression=target.expression.copy()))
                out.append((f'flip_{old_t.__name__}', to_sql(t)))
            except Exception:
                pass

    lits = list(base.find_all(exp.Literal))
    num_positions = [i for i, x in enumerate(lits) if not x.is_string]
    for pos in num_positions[:2]:
        for delta in [1, -1]:
            t = base.copy()
            lit = list(t.find_all(exp.Literal))[pos]
            try:
                val = float(lit.this) + delta
                new_txt = str(int(val)) if float(val).is_integer() else str(val)
                lit.replace(exp.Literal.number(new_txt))
                out.append(('numeric_literal', to_sql(t)))
            except Exception:
                pass

    str_positions = [i for i, x in enumerate(lits) if x.is_string]
    for pos in str_positions[:2]:
        t = base.copy()
        lit = list(t.find_all(exp.Literal))[pos]
        try:
            lit.replace(exp.Literal.string(str(lit.this) + '__ALT__'))
            out.append(('string_literal', to_sql(t)))
        except Exception:
            pass

    agg_swaps = {
        exp.Max: exp.Min,
        exp.Min: exp.Max,
        exp.Avg: exp.Max,
        exp.Sum: exp.Avg,
    }
    for old_t, new_t in agg_swaps.items():
        nodes = list(base.find_all(old_t))
        if nodes:
            t = base.copy()
            target = list(t.find_all(old_t))[0]
            try:
                target.replace(new_t(this=target.this.copy()))
                out.append((f'agg_{old_t.__name__}_to_{new_t.__name__}', to_sql(t)))
            except Exception:
                pass

    sel = base if isinstance(base, exp.Select) else None
    if sel is not None and sel.args.get('where') is not None:
        t = base.copy()
        t.set('where', None)
        out.append(('remove_where', to_sql(t)))

    # Removing gold DISTINCT is safer than adding DISTINCT to a query whose
    # natural-language duplicate semantics may be ambiguous.
    if sel is not None and sel.args.get('distinct'):
        t = base.copy()
        t.set('distinct', None)
        out.append(('remove_distinct', to_sql(t)))

    if sel is not None and sel.args.get('limit') is not None:
        t = base.copy()
        t.set('limit', None)
        out.append(('remove_limit', to_sql(t)))

    dedup, seen = [], {sql.strip().rstrip(';')}
    for typ, s in out:
        key = s.strip().rstrip(';')
        if key not in seen:
            seen.add(key)
            dedup.append((typ, s))
    return dedup

def _stable_seed(text):
    h = hashlib.sha256(text.encode('utf-8')).hexdigest()
    return CFG['seed'] + int(h[:8], 16)

def valid_mutation(ex):
    if CFG['require_outcome_flexible'] and not outcome_flexible(ex['gold_sql']):
        return None

    gold = execute_sql(
        ex['db_path'], ex['gold_sql'],
        CFG['gold_timeout_s'], CFG['max_result_rows']
    )
    if not gold['ok'] or gold['truncated']:
        return None

    muts = mutation_candidates(ex['gold_sql'])
    local_rng = random.Random(_stable_seed(ex['question']))
    local_rng.shuffle(muts)

    for mut_type, mut_sql in muts:
        if CFG['require_outcome_flexible'] and not outcome_flexible(mut_sql):
            continue

        got = execute_sql(
            ex['db_path'], mut_sql,
            CFG['mutation_timeout_s'], CFG['max_result_rows']
        )
        if not got['ok'] or got['truncated']:
            continue

        eq = results_equivalent(gold, got, ex['gold_sql'], mut_sql)
        if eq is False:
            return {
                'mutation_type': mut_type,
                'wrong_sql': mut_sql,
                'gold_result_n': gold['n_rows'],
                'wrong_result_n': got['n_rows'],
                'gold_actual_empty': int(gold['n_rows'] == 0),
                'wrong_actual_empty': int(got['n_rows'] == 0),
            }
    return None

In [6]:
# ============================================================
# 4B. FAST INTERNAL SMOKE TESTS
# ============================================================
assert outcome_flexible('SELECT name FROM employee WHERE salary > 100') is True
assert outcome_flexible('SELECT COUNT(*) FROM employee') is False
assert outcome_flexible('SELECT dept, COUNT(*) FROM employee GROUP BY dept') is True
assert outcome_flexible('SELECT 1') is False
assert outcome_flexible('SELECT name FROM employee LIMIT 0') is False

_a = {'ok': True, 'rows': [(1, 'a'), (2, 'b')], 'truncated': False}
_b = {'ok': True, 'rows': [(2, 'b'), (1, 'a')], 'truncated': False}
assert results_equivalent(_a, _b, 'SELECT x,y FROM t', 'SELECT x,y FROM t') is True
assert results_equivalent(
    _a, _b,
    'SELECT x,y FROM t ORDER BY x',
    'SELECT x,y FROM t ORDER BY x'
) is False

print('✓ Smoke tests passed: outcome-flexibility + result-equivalence logic')

✓ Smoke tests passed: outcome-flexibility + result-equivalence logic


In [7]:
# ============================================================
# 5. BUILD 50 BASE PAIRS
# ============================================================
pool = EXAMPLES.copy()
random.Random(CFG['seed']).shuffle(pool)

base_rows, seen = [], set()

for ex in tqdm(pool[:CFG['max_candidate_search']], desc='Finding valid base pairs'):
    if len(base_rows) >= CFG['n_base']:
        break

    key = (ex['db_id'], ex['question'])
    if key in seen:
        continue

    mut = valid_mutation(ex)
    if mut is None:
        continue

    try:
        sch = schema_text(ex['db_path'], CFG['max_schema_chars'])
    except Exception:
        continue

    item_id = f"{ex['db_id']}__{len(base_rows):03d}"
    base_rows.append({
        'item_id': item_id,
        'db_id': ex['db_id'],
        'db_path': ex['db_path'],
        'question': ex['question'],
        'evidence': ex['evidence'],
        'schema': sch,
        'gold_sql': ex['gold_sql'],
        **mut,
    })
    seen.add(key)

BASE = pd.DataFrame(base_rows)
print(f"Built {len(BASE)} / {CFG['n_base']} requested base pairs.")

if len(BASE) == 0:
    raise RuntimeError('Could not create any valid mutation pairs.')

if len(BASE) < CFG['n_base']:
    print(
        "WARNING: fewer pairs than requested. Increase CFG['max_candidate_search'] "
        'or use another benchmark before interpreting statistical power.'
    )

display(BASE[[
    'item_id', 'db_id', 'mutation_type',
    'gold_result_n', 'wrong_result_n',
    'gold_actual_empty', 'wrong_actual_empty',
    'question'
]].head(10))

print('\nMutation distribution:')
display(BASE['mutation_type'].value_counts().rename_axis('mutation_type').to_frame('n'))

BASE.to_csv(WORK / 'pilot_base_pairs.csv', index=False)

AUDIT = BASE[[
    'item_id', 'db_id', 'question', 'gold_sql', 'wrong_sql',
    'mutation_type', 'gold_result_n', 'wrong_result_n'
]].copy()
AUDIT['manual_label_ok'] = ''
AUDIT['audit_notes'] = ''
AUDIT.to_csv(WORK / 'pilot_manual_audit.csv', index=False)
print('Manual-audit sheet:', WORK / 'pilot_manual_audit.csv')

Finding valid base pairs:   0%|          | 0/1034 [00:00<?, ?it/s]

Built 50 / 50 requested base pairs.


,item_id,db_id,mutation_type,gold_result_n,wrong_result_n,gold_actual_empty,wrong_actual_empty,question
0,cre_Doc_Template_Mgt__000,cre_Doc_Template_Mgt,remove_where,2,15,0,0,What is the document name and template id for ...
1,world_1__001,world_1,numeric_literal,1,2,0,0,Which language is the most popular on the Asia...
2,world_1__002,world_1,numeric_literal,1,2,0,0,Which language is spoken by the largest number...
3,singer__003,singer,remove_where,7,8,0,0,What are the names of the singers who are not ...
4,flight_2__004,flight_2,remove_limit,1,12,0,0,Find the abbreviation and country of the airli...
5,tvshow__005,tvshow,remove_limit,3,12,0,0,List top 3 highest Rating TV series. List the...
6,flight_2__006,flight_2,flip_EQ,12,0,0,1,What are the airline names and abbreviations f...
7,cre_Doc_Template_Mgt__007,cre_Doc_Template_Mgt,remove_limit,1,12,0,0,What is the id and type code for the template ...
8,cre_Doc_Template_Mgt__008,cre_Doc_Template_Mgt,numeric_literal,9,7,0,0,What is the version number and template type c...
9,flight_2__009,flight_2,flip_EQ,0,100,1,0,Give the city and country for the Alton airport.



Mutation distribution:


,n
mutation_type,
flip_EQ,17
numeric_literal,10
remove_where,9
remove_limit,9
agg_Sum_to_Avg,2
flip_NEQ,1
string_literal,1
flip_GTE,1


Manual-audit sheet: /content/drive/MyDrive/Text2SQL_ExecutionBias_Pilot/pilot_manual_audit.csv


In [8]:
# ============================================================
# 6. EXPAND TO 100 CANDIDATES × 3 CONDITIONS
# ============================================================
candidate_rows = []
for _, r in BASE.iterrows():
    common = {
        'item_id': r['item_id'],
        'db_id': r['db_id'],
        'db_path': r['db_path'],
        'question': r['question'],
        'evidence': r['evidence'],
        'schema': r['schema'],
        'gold_sql': r['gold_sql'],
        'mutation_type': r['mutation_type'],
    }

    candidate_rows.append({
        **common,
        'candidate_id': r['item_id'] + '__correct',
        'true_status': 'correct',
        'candidate_sql': r['gold_sql'],
    })

    candidate_rows.append({
        **common,
        'candidate_id': r['item_id'] + '__wrong',
        'true_status': 'wrong',
        'candidate_sql': r['wrong_sql'],
    })

CAND = pd.DataFrame(candidate_rows)

FEEDBACK = {
    'none': '',
    'empty': 'Execution status: SUCCESS.\nResult cardinality: 0 rows.',
    'nonempty': 'Execution status: SUCCESS.\nResult cardinality: >0 rows.',
}

trial_rows = []
for _, r in CAND.iterrows():
    for cond in CFG['conditions']:
        trial_rows.append({
            **r.to_dict(),
            'condition': cond,
            'feedback_text': FEEDBACK[cond],
        })

TRIALS = pd.DataFrame(trial_rows)

if CFG['shuffle_trials']:
    TRIALS = TRIALS.sample(frac=1.0, random_state=CFG['seed']).reset_index(drop=True)

print('Base questions :', len(BASE))
print('Candidates     :', len(CAND))
print('Trials / model :', len(TRIALS))
display(TRIALS.groupby(['true_status', 'condition']).size().unstack(fill_value=0))

TRIALS.to_csv(WORK / 'pilot_trials.csv', index=False)

Base questions : 50
Candidates     : 100
Trials / model : 300


condition,empty,none,nonempty
true_status,,,
correct,50,50,50
wrong,50,50,50


In [9]:
# ============================================================
# 7. VERIFICATION PROMPT
# ============================================================
SYSTEM_PROMPT = """You are a careful SQL verifier.
Determine whether the candidate SQLite query correctly answers the natural-language question for the given database schema.
If the candidate is incorrect, provide a corrected SQL query.
Return only valid JSON with exactly these keys:
{"verdict":"CORRECT or INCORRECT","revised_sql":"SQL string or empty string"}
"""

def make_user_prompt(r):
    ev = ''
    if str(r.get('evidence', '')).strip():
        ev = f"\nAdditional context/evidence:\n{r['evidence']}\n"

    # Truly omit the execution block for the control condition.
    fb = ''
    if r['condition'] != 'none':
        fb = f"\nExecution observation:\n{r['feedback_text']}\n"

    return f"""Natural-language question:
{r['question']}
{ev}
SQLite schema:
{r['schema']}

Candidate SQL:
{r['candidate_sql']}
{fb}
Decide whether the candidate SQL correctly answers the question.
If it is incorrect, provide a corrected SQLite query.
Return JSON only."""

print(make_user_prompt(TRIALS.iloc[0])[:2500])

Natural-language question:
Which airline has most number of flights?

SQLite schema:
TABLE airlines (uid INTEGER PRIMARY KEY, Airline TEXT, Abbreviation TEXT, Country TEXT);
TABLE airports (City TEXT, AirportCode TEXT PRIMARY KEY, AirportName TEXT, Country TEXT, CountryAbbrev TEXT);
TABLE flights (Airline INTEGER PRIMARY KEY, FlightNo INTEGER PRIMARY KEY, SourceAirport TEXT, DestAirport TEXT);
FOREIGN KEY flights.DestAirport -> airports.AirportCode
FOREIGN KEY flights.SourceAirport -> airports.AirportCode

Candidate SQL:
SELECT T1.Airline FROM AIRLINES AS T1 JOIN FLIGHTS AS T2 ON T1.uid = T2.Airline GROUP BY T1.Airline ORDER BY COUNT(*) DESC

Execution observation:
Execution status: SUCCESS.
Result cardinality: >0 rows.

Decide whether the candidate SQL correctly answers the question.
If it is incorrect, provide a corrected SQLite query.
Return JSON only.


In [ ]:
# ============================================================
# 8. MODEL ADAPTER
# ============================================================
import gc, re, json, torch
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoProcessor, BitsAndBytesConfig

def preferred_dtype():
    if not torch.cuda.is_available():
        return torch.float32
    major, _ = torch.cuda.get_device_capability(0)
    return torch.bfloat16 if major >= 8 else torch.float16

def make_quant_config():
    if not (CFG["load_in_4bit"] and torch.cuda.is_available()):
        return None
    return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=preferred_dtype(),
    )

class TextModel:
    def __init__(self, model_id):
        self.model_id = model_id
        self.kind = "causal"
        self.tokenizer = None
        self.processor = None

        kwargs = {
            "device_map": "auto" if torch.cuda.is_available() else None,
            "torch_dtype": preferred_dtype(),
            "trust_remote_code": CFG["trust_remote_code"],
        }
        qc = make_quant_config()
        if qc is not None:
            kwargs["quantization_config"] = qc
        if HF_TOKEN:
            kwargs["token"] = HF_TOKEN

        if "gemma-3-" in model_id.lower():
            from transformers import Gemma3ForConditionalGeneration
            self.kind = "gemma3"
            self.processor = AutoProcessor.from_pretrained(
                model_id, token=HF_TOKEN if HF_TOKEN else None
            )
            self.model = Gemma3ForConditionalGeneration.from_pretrained(
                model_id, **kwargs
            ).eval()
        else:
            self.tokenizer = AutoTokenizer.from_pretrained(
                model_id,
                token=HF_TOKEN if HF_TOKEN else None,
                trust_remote_code=CFG["trust_remote_code"],
            )
            self.model = AutoModelForCausalLM.from_pretrained(
                model_id, **kwargs
            ).eval()

    @property
    def device(self):
        try:
            return self.model.get_input_embeddings().weight.device
        except Exception:
            return next(self.model.parameters()).device

    def generate(self, system_prompt, user_prompt):
        if self.kind == "gemma3":
            messages = [
                {"role": "system", "content": [{"type": "text", "text": system_prompt}]},
                {"role": "user", "content": [{"type": "text", "text": user_prompt}]},
            ]
            inputs = self.processor.apply_chat_template(
                messages,
                add_generation_prompt=True,
                tokenize=True,
                return_dict=True,
                return_tensors="pt",
            )
            inputs = {
                k: v.to(self.device) if hasattr(v, "to") else v
                for k, v in inputs.items()
            }
            in_len = inputs["input_ids"].shape[-1]
            gen_kw = {
                "max_new_tokens": CFG["max_new_tokens"],
                "do_sample": CFG["temperature"] > 0,
            }
            if CFG["temperature"] > 0:
                gen_kw["temperature"] = CFG["temperature"]
            with torch.inference_mode():
                out = self.model.generate(**inputs, **gen_kw)
            return self.processor.decode(
                out[0][in_len:], skip_special_tokens=True
            ).strip()

        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ]
        kw = {
            "add_generation_prompt": True,
            "tokenize": True,
            "return_dict": True,
            "return_tensors": "pt",
        }
        if "qwen3" in self.model_id.lower():
            kw["enable_thinking"] = False

        try:
            inputs = self.tokenizer.apply_chat_template(messages, **kw)
        except TypeError:
            kw.pop("enable_thinking", None)
            inputs = self.tokenizer.apply_chat_template(messages, **kw)

        inputs = {k: v.to(self.device) for k, v in inputs.items()}
        in_len = inputs["input_ids"].shape[-1]

        gen_kw = {
            "max_new_tokens": CFG["max_new_tokens"],
            "do_sample": CFG["temperature"] > 0,
            "pad_token_id": self.tokenizer.eos_token_id,
        }
        if CFG["temperature"] > 0:
            gen_kw["temperature"] = CFG["temperature"]

        with torch.inference_mode():
            out = self.model.generate(**inputs, **gen_kw)

        return self.tokenizer.decode(
            out[0][in_len:], skip_special_tokens=True
        ).strip()

    def close(self):
        del self.model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


def clean_sql_text(x):
    if x is None:
        return ''
    try:
        if pd.isna(x):
            return ''
    except Exception:
        pass

    s = str(x).strip()
    if s.startswith('```'):
        s = re.sub(r'^```(?:sql|sqlite)?\s*', '', s, flags=re.I)
        s = re.sub(r'\s*```$', '', s)
    s = re.sub(r'^\s*(?:SQL|REVISED SQL)\s*:\s*', '', s, flags=re.I)
    return s.strip()

def parse_output(text):
    raw = text.strip()
    obj = None

    candidates = [raw] + re.findall(r"\{.*?\}", raw, flags=re.S)
    for c in candidates:
        try:
            obj = json.loads(c)
            break
        except Exception:
            pass

    verdict = None
    revised = ""

    if isinstance(obj, dict):
        v = str(obj.get("verdict", "")).upper()
        if "INCORRECT" in v:
            verdict = "INCORRECT"
        elif "CORRECT" in v:
            verdict = "CORRECT"
        revised = clean_sql_text(obj.get("revised_sql", ""))

    if verdict is None:
        upper = raw.upper()
        if "INCORRECT" in upper:
            verdict = "INCORRECT"
        elif "CORRECT" in upper:
            verdict = "CORRECT"

    return {
        "parsed_verdict": verdict,
        "revised_sql": revised,
        "parse_ok": verdict in {"CORRECT", "INCORRECT"},
    }

## 3. Running the pilot

Results are written to disk every 10 trials. If the runtime disconnects, rerun the notebook: already completed `(model, candidate, condition)` rows are skipped.

The two models are loaded **sequentially**, so VRAM is not shared between them.

In [ ]:
# ============================================================
# 9. RUN — RESUMABLE + FALLBACK MODEL
# ============================================================
RESULT_PATH = WORK / 'pilot_model_results.csv'

if RESULT_PATH.exists() and not CFG['overwrite_results']:
    RESULTS = pd.read_csv(RESULT_PATH)
    print('Resuming:', len(RESULTS), 'existing rows')
else:
    RESULTS = pd.DataFrame()

def completed_keys(df):
    if df.empty:
        return set()
    return set(zip(
        df['model_id'].astype(str),
        df['candidate_id'].astype(str),
        df['condition'].astype(str),
    ))

def successful_models(df):
    if df.empty:
        return set()
    stats = (
        df.groupby('model_id')['parse_ok']
        .agg(['count', 'mean'])
    )
    # Count a model as usable only after it has essentially completed the trial set
    # and produced structured outputs reliably.
    good = stats[
        (stats['count'] >= 0.95 * len(TRIALS))
        & (stats['mean'] >= 0.90)
    ]
    return set(good.index.astype(str))

def run_one_model(model_id, RESULTS):
    completed = completed_keys(RESULTS)
    remaining = TRIALS[
        ~TRIALS.apply(
            lambda r: (model_id, r['candidate_id'], r['condition']) in completed,
            axis=1
        )
    ]

    if remaining.empty:
        print('Already complete:', model_id)
        return RESULTS

    print('\n' + '=' * 88)
    print('Loading:', model_id)
    print('Remaining:', len(remaining))

    try:
        backend = TextModel(model_id)
    except Exception as e:
        print('MODEL LOAD FAILED:', model_id)
        print(type(e).__name__, str(e)[:1200])
        if CFG['skip_model_on_load_error']:
            print('Skipping model.')
            return RESULTS
        raise

    buffer = []
    try:
        for _, r in tqdm(remaining.iterrows(), total=len(remaining), desc=model_id):
            t0 = time.time()
            err = None
            try:
                raw = backend.generate(SYSTEM_PROMPT, make_user_prompt(r))
            except Exception as e:
                raw = ''
                err = f'{type(e).__name__}: {e}'

            parsed = parse_output(raw)
            buffer.append({
                'model_id': model_id,
                'candidate_id': r['candidate_id'],
                'item_id': r['item_id'],
                'db_id': r['db_id'],
                'db_path': r['db_path'],
                'true_status': r['true_status'],
                'mutation_type': r['mutation_type'],
                'condition': r['condition'],
                'question': r['question'],
                'candidate_sql': r['candidate_sql'],
                'gold_sql': r['gold_sql'],
                'raw_output': raw,
                'latency_s': time.time() - t0,
                'error': err,
                **parsed,
            })

            if len(buffer) >= 10:
                RESULTS = pd.concat([RESULTS, pd.DataFrame(buffer)], ignore_index=True)
                RESULTS.to_csv(RESULT_PATH, index=False)
                buffer = []

        if buffer:
            RESULTS = pd.concat([RESULTS, pd.DataFrame(buffer)], ignore_index=True)
            RESULTS.to_csv(RESULT_PATH, index=False)
    finally:
        backend.close()

    return RESULTS

for model_id in CFG['models']:
    RESULTS = run_one_model(model_id, RESULTS)

for model_id in CFG['fallback_models']:
    if len(successful_models(RESULTS)) >= CFG['minimum_successful_models']:
        break
    RESULTS = run_one_model(model_id, RESULTS)

print('\nTotal result rows:', len(RESULTS))
print('Models with parsed outputs:', sorted(successful_models(RESULTS)))

if not RESULTS.empty:
    display(
        RESULTS.groupby(['model_id', 'true_status', 'condition'])['parse_ok']
        .agg(['count', 'mean'])
    )

In [ ]:
# ============================================================
# 10. OPTIONAL SECONDARY OUTCOME: REVISED-SQL EXECUTION
# ============================================================
# Primary RQ = judgment shift. This secondary outcome evaluates whether
# a generated revision actually repairs or harms the candidate on the
# original database instance.

def check_revision(row):
    if not bool(row.get('parse_ok', False)):
        return None, 'parse_fail'

    if row['parsed_verdict'] == 'CORRECT':
        return row['true_status'] == 'correct', 'kept_candidate'

    revised = clean_sql_text(row.get('revised_sql', ''))
    if not revised:
        return False, 'no_revision'

    gold = execute_sql(
        row['db_path'], row['gold_sql'],
        CFG['gold_timeout_s'], CFG['max_result_rows']
    )
    pred = execute_sql(
        row['db_path'], revised,
        CFG['mutation_timeout_s'], CFG['max_result_rows']
    )

    if not gold['ok']:
        return None, 'gold_exec_error'
    if not pred['ok']:
        return False, 'revision_exec_error'
    if gold['truncated'] or pred['truncated']:
        return None, 'truncated'

    eq = results_equivalent(gold, pred, row['gold_sql'], revised)
    if eq is None:
        return None, 'comparison_unknown'
    return bool(eq), 'exec_compared'

if not RESULTS.empty:
    checked = RESULTS.apply(check_revision, axis=1)
    RESULTS['post_exec_correct'] = [x[0] for x in checked]
    RESULTS['post_exec_note'] = [x[1] for x in checked]
    RESULTS['harmful_overcorrection'] = (
        (RESULTS['true_status'] == 'correct')
        & (RESULTS['parsed_verdict'] == 'INCORRECT')
        & (RESULTS['post_exec_correct'] == False)
    )
    RESULTS.to_csv(RESULT_PATH, index=False)

if not RESULTS.empty:
    display(RESULTS[[
        'model_id', 'candidate_id', 'true_status', 'condition',
        'parsed_verdict', 'post_exec_correct', 'harmful_overcorrection', 'post_exec_note'
    ]].head())

In [ ]:
# ============================================================
# 11. PRIMARY PAIRED ANALYSIS
# ============================================================
from scipy.stats import binomtest

R = RESULTS[RESULTS['parse_ok'] == True].copy()
R['judge_incorrect'] = (R['parsed_verdict'] == 'INCORRECT').astype(int)
R['judge_correct'] = (R['parsed_verdict'] == 'CORRECT').astype(int)
R['judgment_accurate'] = (
    ((R['true_status'] == 'correct') & (R['parsed_verdict'] == 'CORRECT'))
    |
    ((R['true_status'] == 'wrong') & (R['parsed_verdict'] == 'INCORRECT'))
).astype(int)

agg_spec = {
    'n': ('candidate_id', 'count'),
    'p_judge_incorrect': ('judge_incorrect', 'mean'),
    'judgment_accuracy': ('judgment_accurate', 'mean'),
}
if 'post_exec_correct' in R.columns:
    agg_spec['post_exec_accuracy'] = ('post_exec_correct', 'mean')
if 'harmful_overcorrection' in R.columns:
    agg_spec['harmful_overcorrection_rate'] = ('harmful_overcorrection', 'mean')

SUMMARY = (
    R.groupby(['model_id', 'true_status', 'condition'])
     .agg(**agg_spec)
     .reset_index()
)
display(SUMMARY)

def paired_test(df, status, cond_a, cond_b, outcome):
    z = df[df['true_status'] == status]
    piv = z.pivot_table(
        index='candidate_id', columns='condition', values=outcome, aggfunc='first'
    )
    if cond_a not in piv.columns or cond_b not in piv.columns:
        return None

    p = piv[[cond_a, cond_b]].dropna()
    if len(p) == 0:
        return None

    a = p[cond_a].astype(int)
    b = p[cond_b].astype(int)
    a1b0 = int(((a == 1) & (b == 0)).sum())
    a0b1 = int(((a == 0) & (b == 1)).sum())
    discord = a1b0 + a0b1
    pval = binomtest(a1b0, discord, 0.5).pvalue if discord else 1.0

    return {
        'n_pairs': len(p),
        'rate_a': a.mean(),
        'rate_b': b.mean(),
        'paired_diff_a_minus_b': (a - b).mean(),
        'a1_b0': a1b0,
        'a0_b1': a0b1,
        'exact_mcnemar_p': pval,
    }

effect_rows = []

for model_id, d in R.groupby('model_id'):
    for a, b, label in [
        ('empty', 'none', 'CORRECT SQL: empty vs none — rejection shift'),
        ('empty', 'nonempty', 'CORRECT SQL: empty vs nonempty — rejection shift'),
    ]:
        x = paired_test(d, 'correct', a, b, 'judge_incorrect')
        if x:
            effect_rows.append({'model_id': model_id, 'effect': label, **x})

    for a, b, label in [
        ('nonempty', 'none', 'WRONG SQL: nonempty vs none — false reassurance'),
        ('nonempty', 'empty', 'WRONG SQL: nonempty vs empty — false reassurance'),
    ]:
        x = paired_test(d, 'wrong', a, b, 'judge_correct')
        if x:
            effect_rows.append({'model_id': model_id, 'effect': label, **x})

    if 'harmful_overcorrection' in d.columns:
        for a, b, label in [
            ('empty', 'none', 'CORRECT SQL: empty vs none — harmful correction'),
            ('empty', 'nonempty', 'CORRECT SQL: empty vs nonempty — harmful correction'),
        ]:
            x = paired_test(d, 'correct', a, b, 'harmful_overcorrection')
            if x:
                effect_rows.append({'model_id': model_id, 'effect': label, **x})

EFFECTS = pd.DataFrame(effect_rows)
display(EFFECTS)

SUMMARY.to_csv(WORK / 'pilot_summary.csv', index=False)
EFFECTS.to_csv(WORK / 'pilot_effects.csv', index=False)

In [ ]:
# ============================================================
# 12. CORE GO/NO-GO READOUT
# ============================================================
def rate(df, model, status, cond, col):
    x = df[
        (df['model_id'] == model)
        & (df['true_status'] == status)
        & (df['condition'] == cond)
    ]
    return float(x[col].mean()) if len(x) else np.nan

core_rows = []
for m in sorted(R['model_id'].unique()):
    c_empty = rate(R, m, 'correct', 'empty', 'judge_incorrect')
    c_none = rate(R, m, 'correct', 'none', 'judge_incorrect')
    c_non = rate(R, m, 'correct', 'nonempty', 'judge_incorrect')

    w_non = rate(R, m, 'wrong', 'nonempty', 'judge_correct')
    w_none = rate(R, m, 'wrong', 'none', 'judge_correct')
    w_empty = rate(R, m, 'wrong', 'empty', 'judge_correct')

    row = {
        'model': m,
        'P(reject correct | empty)': c_empty,
        'P(reject correct | none)': c_none,
        'P(reject correct | nonempty)': c_non,
        'EMPTY rejection shift vs none': c_empty - c_none,
        'P(accept wrong | nonempty)': w_non,
        'P(accept wrong | none)': w_none,
        'P(accept wrong | empty)': w_empty,
        'NONEMPTY false-reassurance shift vs none': w_non - w_none,
    }

    if 'harmful_overcorrection' in R.columns:
        h_empty = rate(R, m, 'correct', 'empty', 'harmful_overcorrection')
        h_none = rate(R, m, 'correct', 'none', 'harmful_overcorrection')
        row['P(harmful correction | correct, empty)'] = h_empty
        row['harmful correction shift vs none'] = h_empty - h_none

    core_rows.append(row)

CORE = pd.DataFrame(core_rows)
display(CORE)
CORE.to_csv(WORK / 'pilot_core_readout.csv', index=False)

print("""\nPrimary causal judgment signals:
  > 0 EMPTY rejection shift vs none
      Empty feedback makes CORRECT SQL more likely to be judged incorrect.

  > 0 NONEMPTY false-reassurance shift vs none
      Non-empty feedback makes WRONG SQL more likely to be accepted.

Secondary behavioral signal:
  > 0 harmful correction shift vs none
      Empty feedback causes more actually harmful revisions of correct SQL.

For the pilot, prioritize paired direction consistency, exact McNemar tests,
replication across model families, no-feedback verification above chance,
and parse success close to 100%.\n""")

In [ ]:
# ============================================================
# 13. SANITY CHECKS + TREATMENT-INDUCED FLIPS
# ============================================================
print('Parse success:')
display(
    RESULTS.groupby(['model_id', 'condition'])['parse_ok']
    .agg(['count', 'mean'])
    .rename(columns={'mean': 'parse_success_rate'})
)

print('\nNo-feedback baseline verification accuracy:')
display(
    R[R['condition'] == 'none']
    .groupby(['model_id', 'true_status'])['judgment_accurate']
    .agg(['count', 'mean'])
    .rename(columns={'mean': 'accuracy'})
)

print('\nWrong-candidate acceptance by mutation type:')
wrong = R[R['true_status'] == 'wrong']
if len(wrong):
    display(
        wrong.groupby(['model_id', 'mutation_type', 'condition'])
        .agg(n=('candidate_id', 'count'), p_accept_wrong=('judge_correct', 'mean'))
        .reset_index()
        .head(100)
    )

print('\nEMPTY vs NONEMPTY verdict flips:')
piv = R.pivot_table(
    index=['model_id', 'candidate_id', 'true_status', 'question', 'candidate_sql'],
    columns='condition', values='parsed_verdict', aggfunc='first'
).reset_index()

if 'empty' in piv.columns and 'nonempty' in piv.columns:
    flips = piv[piv['empty'] != piv['nonempty']]
    display(flips.head(30))
    print('Total flips:', len(flips))

print('\nAll included candidates passed the outcome-flexibility filter;')
print('plain scalar aggregates such as SELECT COUNT(*) FROM ... are excluded.')

## 4. How to decide whether to continue

The pilot is positive if the **causal pattern** is visible—not because an arbitrary threshold is crossed.

Strong evidence for scaling:

1. **Correct SQL:** `empty` increases incorrect rejection relative to `none` and `nonempty`.
2. **Wrong SQL:** `nonempty` increases erroneous acceptance relative to `none` and `empty`.
3. The effect appears on **paired items**, not only aggregate averages.
4. The direction replicates across model families.
5. Baseline `no feedback` verification is meaningfully above chance.
6. Parse success is high and a quick manual audit confirms the controlled mutations are genuinely wrong.

If only one side is strong, narrow the paper rather than forcing symmetry:

- only the correct-empty side → **empty-induced rejection / over-correction**;
- only the wrong-nonempty side → **non-empty false reassurance**.

If the pilot is positive, the next stage should add a larger set, more model families, another benchmark, a small **real paired-database validation set**, and a simple mitigation such as semantic-first/result-blind verification.

In [ ]:
# ============================================================
# 14. PACKAGE OUTPUTS
# ============================================================
import shutil

archive = shutil.make_archive(
    str(WORK / "execution_bias_pilot_outputs"),
    "zip",
    root_dir=WORK
)

print("Output folder:", WORK.resolve())
print("Archive:", archive)
print("\nKey files:")
for name in [
    "pilot_base_pairs.csv",
    "pilot_manual_audit.csv",
    "pilot_trials.csv",
    "pilot_model_results.csv",
    "pilot_summary.csv",
    "pilot_effects.csv",
    "pilot_core_readout.csv",
]:
    p = WORK / name
    print(("✓" if p.exists() else "·"), p)